# 9.13 现代 PyTorch 推理：SDPA、编译与动态形状导出

> 预估学习时间：45–60 分钟。资料检索：2026-11-17。
> 阅读配套：[PyTorch 2.15 与大模型工程](../../llm_techs/04-PyTorch2.15与大模型工程.md)。
> 实测：2026-11-17，Windows / Python 3.12.9 / PyTorch 2.10.0+cu128，CPU 与 CUDA（RTX 5090 Laptop GPU），后端 aot_eager。

本实验使用小型随机张量，不下载模型、不需要服务端或专用量化包。学习目标：

- 检查因果 Prefill、单 token Decode、分块注意力的掩码与结果。
- 对比 Eager 与编译后的输出，分开测首次调用和预热后延迟。
- 导出带动态形状范围的模型，验证边界、越界与保存 / 加载。
- 计算理论 KV 大小，说明它与实际显存及网络传输的关系。

需要 **PyTorch 2.6+**；这些基础 API 早于 2.15，2.15 发布新增能力见配套文档。默认自动选择 CUDA，否则 CPU；环境变量 `PYTORCH_TUTORIAL_DEVICE=cpu` 可强制 CPU。默认 `aot_eager` 后端，只验证图捕获与结果；在工具链齐备的环境可以设置 `PYTORCH_TUTORIAL_BACKEND=inductor`。后端失败会直接报错，不静默回退。


In [ ]:
import os
import platform
import statistics
import tempfile
from pathlib import Path
from time import perf_counter

import torch
import torch.nn.functional as F

version_pair = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
if version_pair < (2, 6):
    raise RuntimeError("This notebook requires PyTorch >= 2.6.")

requested_device = os.environ.get("PYTORCH_TUTORIAL_DEVICE", "auto")
if requested_device not in {"auto", "cpu", "cuda"}:
    raise ValueError("PYTORCH_TUTORIAL_DEVICE must be auto, cpu, or cuda.")
if requested_device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable.")
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
) if requested_device == "auto" else torch.device(requested_device)

backend = os.environ.get("PYTORCH_TUTORIAL_BACKEND", "aot_eager")
if backend not in {"aot_eager", "inductor"}:
    raise ValueError("Choose aot_eager or inductor for this experiment.")

torch.manual_seed(42)
torch.set_num_threads(min(4, torch.get_num_threads()))
print(f"Python: {platform.python_version()}, PyTorch: {torch.__version__}")
print(f"Device: {device}, dtype: float32, compile backend: {backend}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")
print("Key: record the actual device, dtype, backend, and software versions.")


## 1. 因果 Prefill：先与直接计算对齐

张量形状为 `[batch, heads, tokens, head_dim]`。参考实现显式生成注意力矩阵，仅用于小样本正确性检查；真实长序列应避免这种二次方中间存储。

SDPA 的布尔 mask 中 `True` 表示允许注意力。推理必须显式令 `dropout_p=0.0`；后端根据设备、dtype 与形状自动选择，不能把调用 SDPA 等同于选中了 FlashAttention。


In [ ]:
def attention_reference(q, k, v, allowed):
    scores = (q @ k.transpose(-2, -1)) * (q.shape[-1] ** -0.5)
    scores = scores.masked_fill(~allowed, float("-inf"))
    return scores.softmax(dim=-1) @ v


batch_size, heads, seq_len, head_dim = 2, 4, 12, 16
q = torch.randn(batch_size, heads, seq_len, head_dim, device=device)
k = torch.randn_like(q)
v = torch.randn_like(q)
causal_mask = torch.ones(seq_len, seq_len, device=device, dtype=torch.bool).tril()

with torch.inference_mode():
    reference = attention_reference(q, k, v, causal_mask)
    full_output = F.scaled_dot_product_attention(
        q, k, v, is_causal=True, dropout_p=0.0
    )
torch.testing.assert_close(full_output, reference, rtol=1e-4, atol=1e-5)
print("Prefill maximum absolute error:", (full_output - reference).abs().max().item())
print("Key: establish correctness before timing an optimized implementation.")


## 2. 单 token Decode：非方阵因果掩码的陷阱

如果查询对应序列的最后一个位置，而 KV 只包含有效历史和当前 token，那么所有 key 都应该可见。

SDPA 对非方阵 `is_causal=True` 使用左上对齐的因果掩码：当 Q 长度为 1 时，它只允许第一个 key。下面把正确与错误写法分别与完整因果前向的最后一行比较。这个结论不适用于含 padding、未来槽位或滑动窗口约束的 KV。


In [ ]:
with torch.inference_mode():
    last_query = q[:, :, -1:, :]
    decoded = F.scaled_dot_product_attention(
        last_query, k, v, is_causal=False, dropout_p=0.0
    )
    wrong_alignment = F.scaled_dot_product_attention(
        last_query, k, v, is_causal=True, dropout_p=0.0
    )

expected_last = full_output[:, :, -1:, :]
torch.testing.assert_close(decoded, expected_last, rtol=1e-4, atol=1e-5)
wrong_error = (wrong_alignment - expected_last).abs().max().item()
assert wrong_error > 1e-3, "The fixed example should expose the mask mismatch."
print("Correct decode maximum error:", (decoded - expected_last).abs().max().item())
print("Upper-left mask error:", wrong_error)
print("Key: derive the mask from absolute positions, not just matrix dimensions.")


## 3. 带历史的分块注意力：按绝对位置构造 mask

假设已有 8 个历史 token，再同时计算位置 8–11。第 i 个新查询可见所有历史和它自身之前的新 token。我们构造 `key_position <= query_position`，并与完整因果前向的对应切片比较。


In [ ]:
past_tokens = 8
query_positions = torch.arange(past_tokens, seq_len, device=device)
key_positions = torch.arange(seq_len, device=device)
chunk_mask = key_positions[None, :] <= query_positions[:, None]

with torch.inference_mode():
    chunk_output = F.scaled_dot_product_attention(
        q[:, :, past_tokens:, :],
        k,
        v,
        attn_mask=chunk_mask,
        dropout_p=0.0,
    )
torch.testing.assert_close(
    chunk_output, full_output[:, :, past_tokens:, :], rtol=1e-4, atol=1e-5
)
print("Chunk mask shape:", tuple(chunk_mask.shape))
print("Allowed keys for each new query:", chunk_mask.sum(dim=-1).tolist())
print("Key: chunked attention must preserve the original causal visibility.")


## 4. 编译一个小型投影网络

下面是 LayerNorm + 两层投影与残差，不代表完整 Transformer。编译与 Eager 使用同一份参数和输入。

`fullgraph=True` 让图断裂显式报错；`dynamic=True` 尝试捕获动态形状，不保证未来所有输入只编译一次。默认的 `aot_eager` 用于捕获 / 一致性实验；它不生成 Inductor 融合内核。


In [ ]:
class TinyProjection(torch.nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.norm = torch.nn.LayerNorm(hidden)
        self.up = torch.nn.Linear(hidden, hidden * 2)
        self.down = torch.nn.Linear(hidden * 2, hidden)

    def forward(self, x):
        return x + self.down(F.silu(self.up(self.norm(x))))


model = TinyProjection().to(device).eval()
example_x = torch.randn(2, 32, 64, device=device)
compiled_model = torch.compile(
    model, backend=backend, fullgraph=True, dynamic=True
)


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)


with torch.inference_mode():
    eager_output = model(example_x)
    synchronize()
    started = perf_counter()
    compiled_output = compiled_model(example_x)
    synchronize()
    first_call_ms = (perf_counter() - started) * 1000

torch.testing.assert_close(compiled_output, eager_output, rtol=1e-4, atol=1e-5)
print(f"Compiled first invocation: {first_call_ms:.3f} ms")
print("This includes execution and may reuse disk caches; it is not isolated cold compile time.")
print("Key: fullgraph describes capture boundaries, not the number of GPU kernels.")


## 5. 预热与稳态：报告实际测量口径

每次试验连续运行 10 次前向，以整段同步后的耗时求平均；共进行 7 次试验，报告这些平均值的中位数。CUDA 前后同步，避免仅测到异步提交时间。

这个数包含 Python 调用等开销，不是纯 GPU kernel 时间，也不是服务 TTFT。固定小网络不保证编译更快，所以不对速度做通过 / 失败断言。


In [ ]:
@torch.inference_mode()
def median_forward_ms(fn, x, warmup=3, trials=7, iterations=10):
    for _ in range(warmup):
        fn(x)
    synchronize()
    measurements = []
    for _ in range(trials):
        synchronize()
        started = perf_counter()
        for _ in range(iterations):
            fn(x)
        synchronize()
        measurements.append((perf_counter() - started) * 1000 / iterations)
    return statistics.median(measurements)


eager_ms = median_forward_ms(model, example_x)
compiled_ms = median_forward_ms(compiled_model, example_x)
print(f"Eager: {eager_ms:.4f} ms, {backend}: {compiled_ms:.4f} ms")
if backend == "inductor":
    print(f"Measured eager / inductor latency ratio: {eager_ms / compiled_ms:.3f}")
else:
    print("aot_eager validates captured execution; do not report this as fused-kernel acceleration.")

with torch.inference_mode():
    for token_count in (16, 32, 64):
        changed_x = torch.randn(2, token_count, 64, device=device)
        torch.testing.assert_close(
            compiled_model(changed_x), model(changed_x), rtol=1e-4, atol=1e-5
        )
print("Key: measure speed and verify multiple shapes as separate checks.")


## 6. 动态导出：范围、非法输入与文件往返

导出部分独立在 CPU 上进行，让没有 GPU 的环境也能验证交付约定。输入 `x` 为 `[batch, tokens, 64]`；batch 范围 1–8，tokens 范围 2–128，hidden 固定为 64。

`ExportedProgram` 是带约束的程序表示。这里用 `ep.module()` 检查行为，不代表已部署到端侧运行时或其他推理引擎。


In [ ]:
cpu_model = TinyProjection().eval()
export_x = torch.randn(2, 16, 64)
batch_dim = torch.export.Dim("batch", min=1, max=8)
token_dim = torch.export.Dim("tokens", min=2, max=128)

exported = torch.export.export(
    cpu_model,
    (export_x,),
    dynamic_shapes={"x": {0: batch_dim, 1: token_dim}},
)
exported_module = exported.module()
with torch.inference_mode():
    for shape in ((1, 2, 64), (2, 16, 64), (8, 128, 64), (3, 64, 64)):
        sample = torch.randn(*shape)
        torch.testing.assert_close(
            exported_module(sample), cpu_model(sample), rtol=1e-4, atol=1e-5
        )
        print("Accepted shape:", shape)

print("Exported ranges:", exported.range_constraints)
print("Key: dynamic shapes specify a contract with bounds, not unlimited sizes.")


In [ ]:
for bad_shape in ((9, 16, 64), (2, 129, 64), (2, 16, 63)):
    try:
        with torch.inference_mode():
            exported_module(torch.randn(*bad_shape))
    except (RuntimeError, AssertionError):
        print("Rejected shape:", bad_shape)
    else:
        raise AssertionError(f"Expected shape rejection: {bad_shape}")

with tempfile.TemporaryDirectory(prefix="pytorch_export_") as directory:
    export_path = Path(directory) / "tiny_projection.pt2"
    torch.export.save(exported, export_path)
    restored = torch.export.load(export_path).module()
    with torch.inference_mode():
        torch.testing.assert_close(
            restored(export_x), cpu_model(export_x), rtol=1e-4, atol=1e-5
        )
    print("Export save/load round trip passed; temporary artifact cleaned up.")

print("Key: verify accepted inputs, rejected inputs, and serialized artifacts.")


## 7. 从张量验证走向容量估算

对普通 MHA/GQA 的未压缩 KV，理论字节数为：

`2 × layers × kv_heads × head_dim × tokens × dtype_bytes`

它不包含分页取整、临时工作区、模型权重、分配器保留或传输协议开销；MLA、量化、张量分片和前缀共享还会改变存储或每卡占用。


In [ ]:
layers, kv_heads, kv_head_dim = 32, 8, 128
context_tokens, dtype_bytes = 8192, 2
kv_bytes = 2 * layers * kv_heads * kv_head_dim * context_tokens * dtype_bytes
assert kv_bytes == 1024**3

assumed_link_gb_per_s = 25
transfer_lower_bound_ms = kv_bytes / (assumed_link_gb_per_s * 10**9) * 1000
print(f"Theoretical unsharded KV: {kv_bytes / 1024**3:.3f} GiB per request")
print(f"Assumed-link transfer lower bound: {transfer_lower_bound_ms:.2f} ms")
print("The bandwidth is an assumption, not a measurement.")
print("Key: compare KV transfer overhead with saved interference before adopting PD separation.")


## 8. 思考题

1. 如果用预分配 KV，尚未写入的槽位应该如何屏蔽？为什么不能总用 `is_causal=False`？
2. `fullgraph=True` 与单 kernel 执行有什么不同？
3. 同一 notebook 第二次运行的首次调用明显更快，哪些缓存可能解释这个现象？
4. 导出时只用过 `[2, 16, 64]`，为什么还要验证形状边界和越界？
5. 如何把本实验扩展成服务评测？需要补哪些 TTFT、ITL、质量与工作负载记录？

服务实践见 [推理服务与调度](../../llm_techs/03-推理服务与调度.md)。

## 官方参考

资料检索日期均为 2026-11-17；版本专属更新与基础实验的适用范围已分开标注。

- [PyTorch 2.15 发布说明](https://pytorch.org/blog/pytorch-2-15-release-blog/)
- [SDPA：掩码与 dropout 语义](https://docs.pytorch.org/docs/2.15/generated/torch.nn.functional.scaled_dot_product_attention.html)
- [torch.compile API](https://docs.pytorch.org/docs/2.15/generated/torch.compile.html)
- [Regional compilation](https://docs.pytorch.org/tutorials/recipes/regional_compilation.html)
- [torch.export API 与动态维度](https://docs.pytorch.org/docs/2.15/user_guide/torch_compiler/export/api_reference.html)
